In [1]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [6]:
# TODO: Read image data, feed into Claude
with open("images/prop7.png","rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode('utf-8')

messages = []

add_user_message(messages, [
    # Image Block
    {
        "type": "image",
        "source" : {
            "type": "base64",
            "media_type":"image/png",
            "data": image_bytes
        }
    },
    # Text Block
    {
        "type":"text",
        "text":prompt
    }
])

chat(messages)

Message(id='msg_0157J9BcvJXNrRUY922Z8SGh', container=None, content=[TextBlock(citations=None, text="# Satellite Image Fire Risk Analysis\n\n**1. Residence Identification:**\nThe primary residence is a single-story structure with a light gray/tan shingled roof located in the center of the image, featuring an L-shaped or irregular footprint with what appears to be multiple roof sections, surrounded by dense forest vegetation on all sides.\n\n**2. Tree Overhang Analysis:**\nMultiple large trees with dense canopies directly overhang significant portions of the residence's roof, particularly on the eastern and northern sections, with an estimated 50-75% of the total roof area covered or partially covered by overhanging branches and foliage.\n\n**3. Fire Risk Assessment:**\nThe overhanging trees create multiple wildfire vulnerabilities including numerous ember catch points across the roof surface, direct fuel pathways from the surrounding forest canopy to the structure, and what appears to b

In [8]:
with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(
    messages,
    [
        {
            "type": "document",
            "source": {
                "type": "base64",
                "media_type": "application/pdf",
                "data": file_bytes,
            },
        },
        {"type": "text", "text": "Summarize the document in one sentence"},
    ],
)

chat(messages)

Message(id='msg_0191M9xmJBZR9dXW5wiyadf9', container=None, content=[TextBlock(citations=None, text='The document is a Wikipedia article about Earth, describing it as the third planet from the Sun, the only known astronomical object to harbor life, and detailing its physical characteristics, orbital properties, formation approximately 4.5 billion years ago, and its ocean-covered surface that supports a diverse biosphere.', type='text')], model='claude-sonnet-4-5-20250929', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=9625, output_tokens=65, server_tool_use=None, service_tier='standard'), stop_details=None)

In [9]:
with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(
    messages,
    [
        {
            "type": "document",
            "source": {
                "type": "base64",
                "media_type": "application/pdf",
                "data": file_bytes,
            },
            "title": "earth.pdf",
            "citations": { "enabled": True }
        }
    ],
)

chat(messages)

Message(id='msg_01CF83vViJR4iAAxr3wX58FQ', container=None, content=[TextBlock(citations=None, text="I'd be happy to help you with questions about Earth! I have access to a document with detailed information about our planet. What would you like to know?", type='text')], model='claude-sonnet-4-5-20250929', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=11186, output_tokens=35, server_tool_use=None, service_tier='standard'), stop_details=None)